# Перевод с русского на английский (RNN Seq2Seq)

# Imports

In [38]:
import os
import random
import zipfile
import urllib.request
from pathlib import Path

import keras
import numpy as np
import pandas as pd
import transformers
import tensorflow as tf
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [42]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as exc:
        print(exc)
print(gpus)

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


# Dataset

Используются пары предложений Tatoeba в формате `English<TAB>Russian<TAB>attribution` из [rus-eng.zip](https://www.manythings.org/anki/rus-eng.zip) (зеркало/выборка корпуса Tatoeba).

In [55]:
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
ZIP_PATH = DATA_DIR / 'rus-eng.zip'
DATA_URL = 'https://www.manythings.org/anki/rus-eng.zip'
MAX_PAIRS = 120000
MAX_SOURCE_LEN = 32
MAX_TARGET_LEN = 32
BATCH_SIZE = 32

from urllib.error import HTTPError


def download_file(url, path):
    if path.exists() and path.stat().st_size > 0:
        return
    print('Downloading:', url)
    req = urllib.request.Request(
        url, headers={'User-Agent': 'Mozilla/5.0 (compatible; RNN-lab/1.0)'}
    )
    tmp = path.with_suffix(path.suffix + '.part')
    try:
        with urllib.request.urlopen(req, timeout=120) as response, open(tmp, 'wb') as out:
            import shutil
            shutil.copyfileobj(response, out)
        tmp.replace(path)
    finally:
        if tmp.exists():
            tmp.unlink()


def read_language_file(path, column):
    import bz2
    with bz2.open(path, 'rt', encoding='utf-8') as f:
        first = f.readline().rstrip('\n')
    n_columns = len(first.split('\t'))
    text_position = 2 if n_columns >= 3 else 1
    return pd.read_csv(
        path, sep='\t', header=None, usecols=[0, text_position],
        names=['id', column], dtype={'id': 'int64', column: 'string'},
        on_bad_lines='skip'
    )


if ZIP_PATH.exists() and ZIP_PATH.stat().st_size > 0:
    print('Используется локальный архив:', ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH) as archive:
        txt_files = [x for x in archive.namelist() if x.lower().endswith('.txt')]
        if not txt_files:
            raise ValueError('В архиве нет TSV/TXT файла')
        with archive.open(txt_files[0]) as handle:
            raw = pd.read_csv(handle, sep='\t', header=None, usecols=[0, 1],
                              names=['en', 'ru'], dtype=str, on_bad_lines='skip')
else:
    BASE = 'https://downloads.tatoeba.org/exports/per_language'
    paths = {
        'ru': DATA_DIR / 'rus_sentences.tsv.bz2',
        'en': DATA_DIR / 'eng_sentences.tsv.bz2',
        'links': DATA_DIR / 'eng-rus_links.tsv.bz2',
    }
    urls = {
        'ru': f'{BASE}/rus/rus_sentences.tsv.bz2',
        'en': f'{BASE}/eng/eng_sentences.tsv.bz2',
        'links': f'{BASE}/eng/eng-rus_links.tsv.bz2',
    }
    for key in paths:
        download_file(urls[key], paths[key])

    ru_sentences = read_language_file(paths['ru'], 'ru')
    en_sentences = read_language_file(paths['en'], 'en')
    links = pd.read_csv(paths['links'], sep='\t', header=None,
                        names=['en_id', 'ru_id'], usecols=[0, 1],
                        dtype='int64', on_bad_lines='skip')
    raw = (links
           .merge(en_sentences, left_on='en_id', right_on='id', how='inner')
           .drop(columns='id')
           .merge(ru_sentences, left_on='ru_id', right_on='id', how='inner')
           [['en', 'ru']])
    del ru_sentences, en_sentences, links

raw = (raw.dropna().drop_duplicates(subset=['ru', 'en']).copy())
raw['ru'] = raw['ru'].str.strip()
raw['en'] = raw['en'].str.strip()
raw = raw[(raw.ru.str.len().between(2, 100)) & (raw.en.str.len().between(2, 100))]
raw = raw.sample(frac=1, random_state=SEED).reset_index(drop=True)
if MAX_PAIRS is not None:
    raw = raw.head(MAX_PAIRS).copy()
print('Всего пар перед токенизацией:', len(raw))
display(raw.head())

Всего пар перед токенизацией: 120000


,en,ru
0,There's a spelling mistake on the gravestone. ...,На надгробном камне орфографическая ошибка. Ка...
1,I got married yesterday.,Я вчера женился.
2,Can you speak more slowly?,Вы можете говорить помедленнее?
3,I was already asleep.,Я уже спала.
4,We'll feed the kids first.,Мы сначала накормим детей.


## Токенизатор

Исходный шаблон наследует `GPT2Tokenizer`. В стандартном GPT-2 **BOS = EOS**, что мешает корректной остановке декодера. Добавляем три **различных** специальных токена. `add_special_tokens=True` добавляет BOS/EOS. GPT-2 здесь используется **только как токенизатор**; сама нейросеть — случайно инициализированная RNN, не Transformer.

In [56]:
class Tokenizer(transformers.GPT2Tokenizer):
    def build_inputs_with_special_tokens(self, token_ids_0, token_ids_1=None):
        if token_ids_1 is None:
            return [self.bos_token_id, *token_ids_0, self.eos_token_id]
        return [self.bos_token_id, *token_ids_0, self.bos_token_id,
                *token_ids_1, self.eos_token_id]

tokenizer = Tokenizer.from_pretrained('gpt2')
tokenizer.add_special_tokens({
    'bos_token': '<|startoftranslation|>',
    'eos_token': '<|endoftranslation|>',
    'pad_token': '<|padtranslation|>',
})
PAD_ID = tokenizer.pad_token_id
BOS_ID = tokenizer.bos_token_id
EOS_ID = tokenizer.eos_token_id
VOCAB_SIZE = len(tokenizer)
assert len({PAD_ID, BOS_ID, EOS_ID}) == 3
print('Vocab:', VOCAB_SIZE, 'PAD/BOS/EOS:', PAD_ID, BOS_ID, EOS_ID)
print(tokenizer.encode('Привет, мир!', add_special_tokens=True))

Vocab: 50260 PAD/BOS/EOS: 50259 50257 50258
[140, 253, 21169, 18849, 38857, 16843, 20375, 11, 12466, 120, 18849, 21169, 0]


## Подготовка и разделение выборки

In [58]:
def encode_pair(ru: str, en: str):
    src = tokenizer.encode(ru, add_special_tokens=True, truncation=False)
    tgt = tokenizer.encode(en, add_special_tokens=True, truncation=False)
    if len(src) > MAX_SOURCE_LEN or len(tgt) > MAX_TARGET_LEN:
        return None
    return src, tgt

pairs = []
for ru, en in zip(raw['ru'], raw['en']):
    item = encode_pair(ru, en)
    if item is not None:
        pairs.append((ru, en, *item))

indices = np.random.default_rng(SEED).permutation(len(pairs))
n_test = max(1, int(len(indices) * 0.05))
n_val = max(1, int(len(indices) * 0.05))
idx_test, idx_val, idx_train = indices[:n_test], indices[n_test:n_test+n_val], indices[n_test+n_val:]

def make_arrays(indices):
    src = np.full((len(indices), MAX_SOURCE_LEN), PAD_ID, dtype=np.int32)
    dec = np.full((len(indices), MAX_TARGET_LEN - 1), PAD_ID, dtype=np.int32)
    target = np.full_like(dec, PAD_ID)
    weights = np.zeros_like(dec, dtype=np.float32)
    for j, i in enumerate(indices):
        _, _, source_tokens, target_tokens = pairs[int(i)]
        src[j, :len(source_tokens)] = source_tokens
        length = len(target_tokens) - 1
        dec[j, :length] = target_tokens[:-1]
        target[j, :length] = target_tokens[1:]
        weights[j, :length] = 1.0
    return (src, dec), target, weights

def make_tf_dataset(indices, shuffle=False):
    (src, dec), y, mask = make_arrays(indices)
    ds = tf.data.Dataset.from_tensor_slices(((src, dec), y, mask))
    if shuffle:
        ds = ds.shuffle(min(len(indices), 10000), seed=SEED,
                        reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_tf_dataset(idx_train, shuffle=True)
val_ds = make_tf_dataset(idx_val)
test_ds = make_tf_dataset(idx_test)

for (src, dec), y, mask in train_ds.take(1):
    print('Source:', src.shape, '| Decoder input:', dec.shape,
          '| Target:', y.shape, '| Mask:', mask.shape)
    print('Example RU:', tokenizer.decode(src[0].numpy(), skip_special_tokens=True))
    print('Example EN:', tokenizer.decode(y[0].numpy(), skip_special_tokens=True))
    print('Mask sum:', float(tf.reduce_sum(mask[0])))

Source: (32, 32) | Decoder input: (32, 31) | Target: (32, 31) | Mask: (32, 31)
Example RU: Они в ловушке.
Example EN: 're trapped.
Mask sum: 3.0


# Model

In [59]:
def get_model(
    units: int,
    n_tokens: int,
    n_labels: int,
    n_stacks: int = 1,
    bidirectional: bool = False,
    name: str | None = None,
    cell_type: type[keras.layers.Layer] = keras.layers.LSTMCell,
    use_attention: bool = False,
) -> keras.Model:
    """Создаёт RNN encoder-decoder для перевода текста.

    Принимает [source_ids, decoder_input_ids], возвращает logits [B, P, n_labels].
    n_tokens — размер входного словаря; n_labels — размер выходного словаря.
    """
    if cell_type not in (keras.layers.LSTMCell, keras.layers.GRUCell):
        raise ValueError('cell_type должен быть keras.layers.LSTMCell или GRUCell')
    if n_stacks < 1:
        raise ValueError('n_stacks должен быть >= 1')
    is_lstm = cell_type is keras.layers.LSTMCell
    state_count = 2 if is_lstm else 1
    rnn_type = keras.layers.LSTM if is_lstm else keras.layers.GRU
    name = name or 'seq2seq_rnn'

    src = keras.Input(shape=(None,), dtype='int32', name='source_ids')
    dec = keras.Input(shape=(None,), dtype='int32', name='decoder_input_ids')
    embedding_dim = 256
    enc_embedding = keras.layers.Embedding(n_tokens, embedding_dim, mask_zero=False,
                                            name='encoder_embedding')
    dec_embedding = keras.layers.Embedding(n_labels, embedding_dim, mask_zero=False,
                                            name='decoder_embedding')
    src_mask_layer = keras.layers.Lambda(lambda x: tf.not_equal(x, PAD_ID), name='source_mask')
    dec_mask_layer = keras.layers.Lambda(lambda x: tf.not_equal(x, PAD_ID), name='decoder_mask')
    enc_mask = src_mask_layer(src)
    dec_mask = dec_mask_layer(dec)
    x = enc_embedding(src)
    init_states = []
    for layer_id in range(n_stacks):
        enc_layer = rnn_type(units, return_sequences=True, return_state=True,
                             name=f'encoder_{layer_id}')
        if bidirectional:
            result = keras.layers.Bidirectional(enc_layer, name=f'biencoder_{layer_id}')(x, mask=enc_mask)
            x = result[0]
            f_states = result[1:1+state_count]
            b_states = result[1+state_count:1+2*state_count]
            state = [keras.layers.Dense(units, activation='tanh',
                                       name=f'init_{layer_id}_{k}')(
                         keras.layers.Concatenate()([f_states[k], b_states[k]]))
                     for k in range(state_count)]
        else:
            result = enc_layer(x, mask=enc_mask)
            x, state = result[0], list(result[1:])
        init_states.extend(state)
    encoder_outputs = x
    encoder_model = keras.Model(src, [encoder_outputs, *init_states], name='encoder_inference')

    step_tokens = keras.Input(shape=(None,), dtype='int32', name='step_tokens')
    enc_dim = units * (2 if bidirectional else 1)
    step_enc_outputs = keras.Input(shape=(None, enc_dim), dtype='float32', name='enc_outputs')
    step_src_ids = keras.Input(shape=(None,), dtype='int32', name='enc_source_ids')
    step_states = [keras.Input(shape=(units,), dtype='float32', name=f'state_{j}')
                   for j in range(n_stacks * state_count)]
    h = dec_embedding(step_tokens)
    step_dec_mask = dec_mask_layer(step_tokens)
    new_states = []
    for layer_id in range(n_stacks):
        decoder_layer = rnn_type(units, return_sequences=True, return_state=True,
                                 name=f'decoder_{layer_id}')
        old_states = step_states[layer_id * state_count:(layer_id+1) * state_count]
        result = decoder_layer(h, initial_state=old_states, mask=step_dec_mask)
        h = result[0]
        new_states.extend(result[1:])
    if use_attention:
        context = keras.layers.AdditiveAttention(name='additive_attention')(
            [h, step_enc_outputs],
            mask=[step_dec_mask, src_mask_layer(step_src_ids)])
        h = keras.layers.Concatenate(name='attention_concat')([h, context])
    logits = keras.layers.Dense(n_labels, name='token_logits')(h)
    decoder_model = keras.Model(
        [step_tokens, step_enc_outputs, step_src_ids, *step_states],
        [logits, *new_states], name='decoder_inference')

    outputs = decoder_model([dec, encoder_outputs, src, *init_states])[0]
    model = keras.Model([src, dec], outputs, name=name)
    model.encoder_model = encoder_model
    model.decoder_model = decoder_model
    model.rnn_state_count = state_count
    model.use_attention = use_attention
    return model

UNITS = 256
N_STACKS = 1
USE_ATTENTION = True
BIDIRECTIONAL = False
CELL_TYPE = keras.layers.LSTMCell

model = get_model(UNITS, VOCAB_SIZE, VOCAB_SIZE,
                  n_stacks=N_STACKS, bidirectional=BIDIRECTIONAL,
                  cell_type=CELL_TYPE, use_attention=USE_ATTENTION)
model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'decoder_inference' (of type Functional) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


Model: "seq2seq_rnn"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ source_ids          │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_embedding   │ (None, None, 256) │ 12,866,560 │ source_ids[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ source_mask         │ (None, None)      │          0 │ source_ids[0][0]  │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_input_ids   │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_0 (LSTM)    │ [(None, None,     │    525,312 │ encoder_embeddin… │
│                     │ 256), (None,      │            │ source_mask[0][0] │
│                     │ 256), (None,      │            │                   │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inference   │ [(None, None,     │ 39,175,508 │ decoder_input_id… │
│ (Functional)        │ 50260), (None,    │            │ encoder_0[0][0],  │
│                     │ 256), (None,      │            │ source_ids[0][0], │
│                     │ 256)]             │            │ encoder_0[0][1],  │
│                     │                   │            │ encoder_0[0][2],  │
│                     │                   │            │ source_mask[0][0] │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 52,567,380 (200.53 MB)

 Trainable params: 52,567,380 (200.53 MB)

 Non-trainable params: 0 (0.00 B)

# Training

In [ ]:
EPOCHS = 25
LEARNING_RATE = 1e-3

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE, clipnorm=1.0),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy(name='token_accuracy')],
)
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=2,
                                  restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', patience=1, factor=0.5),
]
history = model.fit(train_ds, validation_data=val_ds,
                    epochs=EPOCHS, callbacks=callbacks)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['loss'], label='train')
axes[0].plot(history.history['val_loss'], label='validation')
axes[0].set_title('Token cross-entropy')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[1].plot(history.history['token_accuracy'], label='train')
axes[1].plot(history.history['val_token_accuracy'], label='validation')
axes[1].set_title('Token accuracy (PAD masked)')
axes[1].set_xlabel('Epoch')
axes[1].legend()
plt.tight_layout()
plt.show()

Epoch 1/25
1796/1796 ━━━━━━━━━━━━━━━━━━━━ 378s 209ms/step - loss: 4.4165 - token_accuracy: 0.3151 - val_loss: 3.5148 - val_token_accuracy: 0.4088 - learning_rate: 0.0010
Epoch 2/25
 160/1796 ━━━━━━━━━━━━━━━━━━━━ 5:32 203ms/step - loss: 3.3907 - token_accuracy: 0.4119

# Testing


In [ ]:
def translate(text: str, tokenizer: Tokenizer, model: keras.Model,
              max_len: int = 20) -> str:
    """Авторегрессионный RU-EN перевод через состояния encoder/decoder."""
    source_tokens = tokenizer.encode(text, add_special_tokens=True)
    if len(source_tokens) > MAX_SOURCE_LEN:
        source_tokens = source_tokens[:MAX_SOURCE_LEN-1] + [EOS_ID]
    src = np.full((1, MAX_SOURCE_LEN), PAD_ID, dtype=np.int32)
    src[0, :len(source_tokens)] = source_tokens

    encoded = model.encoder_model(tf.convert_to_tensor(src), training=False)
    enc_outputs = encoded[0]
    states = list(encoded[1:])
    next_token = BOS_ID
    generated = []
    for _ in range(max_len):
        current = tf.constant([[next_token]], dtype=tf.int32)
        prediction = model.decoder_model(
            [current, enc_outputs, tf.convert_to_tensor(src), *states], training=False)
        logits = prediction[0][:, -1, :]
        states = list(prediction[1:])
        next_token = int(tf.argmax(logits, axis=-1)[0].numpy())
        if next_token == EOS_ID:
            break
        if next_token not in (BOS_ID, PAD_ID):
            generated.append(next_token)
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

examples = [
    'Привет!',
    'Как дела?',
    'Я люблю читать Мураками.',
    'Сегодня хорошая погода.',
    'Где находится D корпус?',
    'Спасибо за помощь.',
]
for s in examples:
    print(f'RU: {s}\nEN: {translate(s, tokenizer, model)}\n')

In [ ]:
results = model.evaluate(test_ds, return_dict=True, verbose=1)
print('Test metrics:', results)

N_TRANSLATIONS = 12
print('\nHeld-out examples:')
for i in idx_test[:N_TRANSLATIONS]:
    ru, ref, _, _ = pairs[int(i)]
    pred = translate(ru, tokenizer, model)
    print(f'RU:  {ru}\nREF: {ref}\nPRED:{pred}\n')

try:
    import sacrebleu
    subset = idx_test[:min(100, len(idx_test))]
    references = [pairs[int(i)][1] for i in subset]
    predictions = [translate(pairs[int(i)][0], tokenizer, model) for i in subset]
    chrf = sacrebleu.corpus_chrf(predictions, [references]).score
    print(f'chrF на {len(subset)} тестовых переводах: {chrf:.2f}')
except ImportError:
    print('Для расчёта chrF установите sacrebleu: %pip install sacrebleu')

## Сравнение с моделью без attention

Чтобы проверить вопрос из задания «улучшает ли attention качество?»

In [54]:
# КРИВОЕ

# baseline = get_model(UNITS, VOCAB_SIZE, VOCAB_SIZE,
#                      n_stacks=N_STACKS, bidirectional=BIDIRECTIONAL,
#                      cell_type=CELL_TYPE, use_attention=False, name='seq2seq_no_attention')
# baseline.compile(optimizer=keras.optimizers.Adam(LEARNING_RATE, clipnorm=1.0),
#                  loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
#                  weighted_metrics=[keras.metrics.SparseCategoricalAccuracy(name='token_accuracy')])
# baseline_history = baseline.fit(train_ds, validation_data=val_ds, epochs=EPOCHS,
#                                 callbacks=[keras.callbacks.EarlyStopping(
#                                     monitor='val_loss', patience=2, restore_best_weights=True)])
# print('Baseline test:', baseline.evaluate(test_ds, return_dict=True))

ValueError: `inputs` not connected to `outputs`